# Xenium data plotting: UMAP, composition plots, barplots and spatial plots

This notebook contains the analyses and plotting code used to generate Figure 2 and associated supplementary visualizations.
> **Reproducibility note:** the notebook expects processed spatial/single-cell objects produced by the preceding preprocessing workflow.


## Setup

Imports and plotting dependencies used throughout the figure-generation workflow.


In [ ]:
import h5py
import warnings
import os
import spatialdata_plot
from pathlib import Path
import logging
from matplotlib.patches import Patch
from scipy.stats import ttest_ind, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from matplotlib.font_manager import FontProperties


## Spatial visualizations

Visualize cells and annotations in spatial coordinates.


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import squidpy as sq
import spatialdata as sd
import spatialdata_io as sio


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
import matplotlib as mpl

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42   # TrueType fonts in PDF (Illustrator-friendly)
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42


In [ ]:
# Run `logging.getLogger('fontTools').setLevel` for this analysis step.
logging.getLogger("fontTools").setLevel(logging.WARNING)
# Run `logging.getLogger('fontTools.subset').setLevel` for this analysis step.
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)


In [ ]:
!pwd


## Figure colors and plotting configuration

Define the color mappings and plotting settings used consistently across the main and supplementary figures.


In [ ]:
# Set output directory for Scanpy plots
sc.settings.figdir = "test"


In [ ]:
# Run `sc.settings.set_figure_params` for this analysis step.
sc.settings.set_figure_params(dpi_save=300, transparent=True)


In [ ]:
# Create output directory for the rest of the plots
out_dir = Path("test")
# Run `out_dir.mkdir` for this analysis step.
out_dir.mkdir(parents=True, exist_ok=True)


## Load processed data

Load the processed objects and metadata used to generate the figure panels.


In [ ]:
# Load the processed AnnData object.
combined_adata = sc.read_h5ad("../../../combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells_ReannotatedIfgga4.h5ad")


In [ ]:
# Run `list` for this analysis step.
list(combined_adata.obs["celltype3"].cat.categories)


In [ ]:
# Rename columns for clearer downstream use.
combined_adata.obs["celltype3"] = (
    combined_adata.obs["celltype3"]
    .cat.rename_categories({
        "VCMs": "Basal VCMs",
        "Myh7+ VCMs": "Remodelled VCMs",
        "Ifgga4+ VCMs": "IFN-associated VCMs"
    })
)


In [ ]:
# Rename columns for clearer downstream use.
combined_adata.obs["celltype4"] = (
    combined_adata.obs["celltype4"]
    .cat.rename_categories({
        "VCMs": "Basal VCMs",
        "Myh7+ VCMs": "Remodelled VCMs",
        "Ifgga4+ VCMs": "IFN-associated VCMs"
    })
)


In [ ]:
# Create a colour dictionary
CELL_STATE_COLORS = {
    "Basal VCMs":               "#2C7BB6",  # blue
    "Stressed VCMs":            "#D7191C",  # red
    "Remodelled VCMs":           "#1A9641",  # green
    "IFN-associated VCMs":      "#FFD92F",  # gold
    "FBs":              "#984EA3",  # purple
    "Vasculature ECs":    "#00A6CA",  # cyan
    "Endocardial ECs": "#FDB863",  # light orange
    "Pericytes":                "#A6761D",  # dark gold
    "SMCs":           "#80CDC1",  # mint
    "Myeloid":                 "#FF7A00",  # orange
    "T cells":                  "#7B2CFF",  # purple
    "B cells":                  "#FF4FA3",  # pink
    "Lymphoid":                  "#7B2CFF",  # purple
    "B cells":                  "#FF4FA3",  # pink
    "Epicardium":              "#35978F",  # teal
    "NCs":            "#8073AC",  # violet
    "ACMs":               "#8C510A",  # brown
    "LECs":   "#4DBBD5"  # light blue
}


In [ ]:
# Assign colors according to category order
combined_adata.uns["celltype3_colors"] = [
    CELL_STATE_COLORS[ct]
    for ct in combined_adata.obs["celltype3"].cat.categories
]


In [ ]:
# Assign colors according to category order
combined_adata.uns["celltype4_colors"] = [
    CELL_STATE_COLORS[ct]
    for ct in combined_adata.obs["celltype4"].cat.categories
]


In [ ]:
# Define the order of cell types
order = [
    "Basal VCMs", "IFN-associated VCMs", "Stressed VCMs", "Remodelled VCMs", 
    "FBs", "Vasculature ECs", "Endocardial ECs", "Epicardium", "Myeloid", "T cells", "B cells",
    "Pericytes", "SMCs", "NCs", "ACMs"
     
]


In [ ]:
# Assign the order of cell types to celltype3 obs.
combined_adata.obs["celltype3"] = pd.Categorical(combined_adata.obs["celltype3"], categories = order, ordered = True)


In [ ]:
# Define the desired order of sample types
ordered_types = ['WT', 'BE', 'R636Q']
# Define the desired order of samples
ordered_names = ['WT_rep1', 'WT_rep2', 'WT_rep4', 'WT_rep5', 'BE_rep1', 'BE_rep2', 'BE_rep3', 'BE_rep4', 'PBS_rep1', 'PBS_rep2', 'PBS_rep3', 'PBS_rep4']


In [ ]:
# Compute and store `combined_adata.obs['type']`.
combined_adata.obs['type'] = pd.Categorical(combined_adata.obs['type'], categories=ordered_types, ordered = True)
# Compute and store `combined_adata.obs['name']`.
combined_adata.obs['name'] = pd.Categorical(combined_adata.obs['name'], categories=ordered_names, ordered = True)


In [ ]:
# Save the processed object to disk.
combined_adata.write_h5ad("../../combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells_ReannotatedIfgga4_newcolours.h5ad")


## Cell-type and sample summaries

Calculate group-level summaries used in the figure panels.


In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42

# Set `cell_col` for the following analysis.
cell_col   = "celltype3"
# Set `sample_col` for the following analysis.
sample_col = "name"
# Set `group_col` for the following analysis.
group_col  = "type"

# work from the object as-is (uses the categorical orders you already set)
obs = combined_adata.obs[[cell_col, sample_col, group_col]].copy()

# Compute and store `celltype_order`.
celltype_order = list(obs[cell_col].cat.categories)
# Compute and store `sample_order`.
sample_order   = list(obs[sample_col].cat.categories)
# Compute and store `group_order`.
group_order    = list(obs[group_col].cat.categories)

# proportions per sample (rows sum to 1)
props_sample = (
    pd.crosstab(obs[sample_col], obs[cell_col], normalize="index")
    .reindex(index=sample_order, columns=celltype_order)
    .fillna(0)
)

# mean proportions per group
sample2group = obs.drop_duplicates(sample_col).set_index(sample_col)[group_col]
# Make an independent copy of the selected data.
tmp = props_sample.copy()
# Set `tmp[group_col]` for the following analysis.
tmp[group_col] = sample2group.reindex(sample_order).values

# Group observations for downstream summarization.
group_means = (
    tmp.groupby(group_col, observed=True)[celltype_order]
    .mean()
    .reindex(index=group_order)
)


In [ ]:
# Export the resulting table as a CSV file.
group_means.to_csv(out_dir/"group_means_Xenium_celltypes_Ifgga4reannotated.csv")


In [ ]:
# Export the resulting table as a CSV file.
props_sample.to_csv(out_dir/"samplenoWT3_Xenium_celltypes_Ifgga4reannotated.csv")


## Export figures

Save figure panels or supplementary plots for manuscript assembly.


In [ ]:
# ---- Plot 1: proportions per sample ----

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=(8, 5))

# Select the required subset and store it as `plot_order`.
plot_order = celltype_order[::-1]

# Run `props_sample[plot_order].plot` for this analysis step.
props_sample[plot_order].plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[CELL_STATE_COLORS[ct] for ct in plot_order],
    width=0.9
)

# Set `legend_handles` for the following analysis.
legend_handles = [
    Patch(facecolor=CELL_STATE_COLORS[ct], label=ct)
    for ct in celltype_order
]

# Add the figure legend.
ax.legend(
    handles=legend_handles,
    title="celltype",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

# Run `ax.grid` for this analysis step.
ax.grid(False)
# Label the y-axis.
ax.set_ylabel("Cell type proportion")
# Label the x-axis.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 1)

# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()

# Save the completed figure to disk.
fig.savefig(
    os.path.join(
        out_dir,
        "celltype_proportions_per_sample_Ifgga4reannotated.pdf"
    ),
    dpi=300,
    bbox_inches="tight",
    transparent=True
)

# Display the completed figure.
plt.show()


In [ ]:
# ---- Plot 2: mean proportions per group ----
fig, ax = plt.subplots(figsize=(5, 5))

# Select the required subset and store it as `plot_order`.
plot_order = celltype_order[::-1]

# Run `group_means[plot_order].plot` for this analysis step.
group_means[plot_order].plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[CELL_STATE_COLORS[ct] for ct in plot_order],
    width=0.8
)

# Set `legend_handles` for the following analysis.
legend_handles = [
    Patch(facecolor=CELL_STATE_COLORS[ct], label=ct)
    for ct in celltype_order
]

# Add the figure legend.
ax.legend(
    handles=legend_handles,
    title="celltype",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

# Run `ax.grid` for this analysis step.
ax.grid(False)
# Label the y-axis.
ax.set_ylabel("Mean cell type proportion")
# Label the x-axis.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 1)
# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()
# Save the completed figure to disk.
fig.savefig(os.path.join(out_dir, "celltype_mean_proportions_per_group_Ifgga4reannotated.pdf"),
            dpi=300, bbox_inches="tight", transparent=True)
# Display the completed figure.
plt.show()


## Embedding visualizations

Generate low-dimensional visualizations used to display cell populations and annotations.


In [ ]:
# Plot the UMAP embedding using the selected annotation or feature.
sc.pl.umap(combined_adata, color = "celltype3", 
           save = "UMAP_noWT3_withT_Bcells_reannotatedIfgga4.pdf")


## Subset VCMs

Generate dotplots for the markers in the subset VCMs and barplots for the VCMs subsets


In [ ]:
# Define the values used for VCMs types
celltypes_to_keep = ["Basal VCMs", "IFN-associated VCMs", "Stressed VCMs", "Remodelled VCMs"]

# Make an independent copy of the VCMs
VCMs = combined_adata[combined_adata.obs["celltype3"].isin(celltypes_to_keep)].copy()


In [ ]:
# Define the values used for `genes_by_group`.
genes_by_group = {
    "Basal VCMs": ["Ttn", "Myl2", "Tnnt2", "Ttn_N2B", "Ttn_flanking-exons", 'Camk2d_isoformA'],
    "IFN-associated VCMs": ["F830016B08Rik", "B2m", "Ttn_N2B", "Ttn_flanking-exons", 'Camk2d_isoformA'],
    "Stressed VCMs": ["Nppa", "Ankrd1", "Nppb", "Ttn_N2A", "Ttn_alt-exon", 'Camk2d_isoformB'],
    "Remodelled VCMs": ["Myh7"]
}


In [ ]:
# Define the values used for `genes_by_group`.
genes_by_group = {
    "VCMs": ["Myl2", "Ttn_N2B", "Ttn_flanking-exons"],
    "Ifgga4+ VCMs": ["F830016B08Rik", "B2m", "Ttn_N2B", "Ttn_flanking-exons"],
    "Stressed VCMs": ["Nppa", "Ankrd1", "Nppb", "Ttn_N2A", "Ttn_alt-exon"],
    "Myh7+ VCMs": ["Myh7"]
}


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(VCMs, groupby = 'celltype3', var_names=genes_by_group, standard_scale = 'var', dendrogram=False, save = "dotplot_mygenes_grouped_VCMsspatial_Ifgga4_reannotated.pdf")


In [ ]:
# Run `sc.pl.rank_genes_groups_dotplot` for this analysis step.
sc.pl.rank_genes_groups_dotplot(VCMs, groupby = 'celltype3', var_names=genes_by_group, dendrogram=False, save = "dotplot_mygenes_grouped_VCMsspatial_Ifgga4_reannotated_novarscale.pdf")


In [ ]:
# Run `sc.settings.set_figure_params` for this analysis step.
sc.settings.set_figure_params(dpi_save=300, transparent=True)


In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Set `mpl.rcParams['pdf.fonttype']` for the following analysis.
mpl.rcParams["pdf.fonttype"] = 42
# Set `mpl.rcParams['ps.fonttype']` for the following analysis.
mpl.rcParams["ps.fonttype"]  = 42

# Set `cell_col` for the following analysis.
cell_col   = "celltype3"
# Set `sample_col` for the following analysis.
sample_col = "name"
# Set `group_col` for the following analysis.
group_col  = "type"

# work from the object as-is (uses the categorical orders you already set)
obs = combined_adata.obs[[cell_col, sample_col, group_col]].copy()

# Compute and store `celltype_order`.
celltype_order = list(obs[cell_col].cat.categories)
# Compute and store `sample_order`.
sample_order   = list(obs[sample_col].cat.categories)
# Compute and store `group_order`.
group_order    = list(obs[group_col].cat.categories)

# proportions per sample (rows sum to 1)
props_sample = (
    pd.crosstab(obs[sample_col], obs[cell_col], normalize="index")
    .reindex(index=sample_order, columns=celltype_order)
    .fillna(0)
)

# mean proportions per group
sample2group = obs.drop_duplicates(sample_col).set_index(sample_col)[group_col]
# Make an independent copy of the selected data.
tmp = props_sample.copy()
# Set `tmp[group_col]` for the following analysis.
tmp[group_col] = sample2group.reindex(sample_order).values

# Group observations for downstream summarization.
group_means = (
    tmp.groupby(group_col, observed=True)[celltype_order]
    .mean()
    .reindex(index=group_order)
)


In [ ]:
import matplotlib.pyplot as plt

# Cardiomyocyte subtypes to include

# Define the values used for `cm_celltypes`.
cm_celltypes = [
    "Basal VCMs",
    "IFN-associated VCMs",
    "Stressed VCMs",
    "Remodelled VCMs"
    # add any other cardiomyocyte subtypes here
]

# Keep only cell types present in props_sample

# Set `cm_celltypes` for the following analysis.
cm_celltypes = [
    ct for ct in cm_celltypes
    if ct in props_sample.columns
]

# Assign colors from CELL_STATE_COLORS

# Define the values used for `cm_color_dict`.
cm_color_dict = {
    "Basal VCMs": CELL_STATE_COLORS["Basal VCMs"],
    "IFN-associated VCMs": CELL_STATE_COLORS["IFN-associated VCMs"],
    "Stressed VCMs": CELL_STATE_COLORS["Stressed VCMs"],
    "Remodelled VCMs": CELL_STATE_COLORS["Remodelled VCMs"]
}

# Select cardiomyocyte proportions

# Make an independent copy of the selected data.
props_cm_sample = props_sample[cm_celltypes].copy()

# Rescale within each sample so cardiomyocyte subtypes sum to 100%

# Compute and store `props_cm_sample`.
props_cm_sample = props_cm_sample.div(
    props_cm_sample.sum(axis=1),
    axis=0
).fillna(0)

# Reverse stacking order while keeping legend order unchanged

# Select the required subset and store it as `plot_order`.
plot_order = cm_celltypes[::-1]

# Compute and store `(fig, ax)`.
fig, ax = plt.subplots(figsize=(8, 5))

# Run `props_cm_sample[plot_order].plot` for this analysis step.
props_cm_sample[plot_order].plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[cm_color_dict[ct] for ct in plot_order],
    width=0.9
)

# Set `legend_handles` for the following analysis.
legend_handles = [
    Patch(facecolor=cm_color_dict[ct], label=ct)
    for ct in cm_celltypes
]

# Add the figure legend.
ax.legend(
    handles=legend_handles,
    title="Cardiomyocyte subtype",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

# Run `ax.grid` for this analysis step.
ax.grid(False)
# Label the y-axis.
ax.set_ylabel("Proportion of cardiomyocytes")
# Label the x-axis.
ax.set_xlabel("")
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(0, 1)

# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()

# Save the completed figure to disk.
fig.savefig(
    os.path.join(
        out_dir,
        "cardiomyocyte_subtype_proportions_per_sample.pdf"
    ),
    dpi=300,
    bbox_inches="tight",
    transparent=True
)

# Display the completed figure.
plt.show()


In [ ]:
# Set `adata` for the following analysis.
adata = combined_adata


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1. Define columns and settings
# -----------------------------

# Set `celltype_col` for the following analysis.
celltype_col = "celltype3"        # your cell-state/cell-type annotation
# Set `sample_col` for the following analysis.
sample_col = "name"            # change to your sample/section column in adata.obs
# Set `condition_col` for the following analysis.
condition_col = "type"       # change to WT/DCM/BE column in adata.obs

# Define the values used for `condition_order`.
condition_order = ["WT", "BE", "R636Q"]

# Define the values used for `celltype_order`.
celltype_order = [
    "Basal VCMs",
    "IFN-associated VCMs",
    "Stressed VCMs",
    "FBs",
    "Myeloid",
    "T cells"
]

# Choose colors here
palette = {
    "WT": "#2C7BB6", #"#706498",   # VCMs
    "BE": "#FFD92F", #"#89375B",    # Ifgga4+ VCMs
    "R636Q": "#D7191C", #"#323067"  # Stressed
     
    
}


In [ ]:
# -----------------------------
# 2. Calculate cell-state composition
# -----------------------------

# Make an independent copy of the selected data.
oobs = adata.obs[[sample_col, condition_col, celltype_col]].copy()

# Count cells per real sample/section, condition, and cell type
counts = (
    obs
    .groupby([sample_col, condition_col, celltype_col], observed=True)
    .size()
    .reset_index(name="n_cells")
)

# Total cells per real sample/section
totals = (
    obs
    .groupby([sample_col, condition_col], observed=True)
    .size()
    .reset_index(name="total_cells")
)

# Keep only real sample-condition combinations
sample_info = totals[[sample_col, condition_col, "total_cells"]].drop_duplicates()

# Add all cell types only within the correct condition of each real sample
celltype_df = pd.DataFrame({celltype_col: celltype_order})

# Compute and store `template`.
template = (
    sample_info.assign(key=1)
    .merge(celltype_df.assign(key=1), on="key")
    .drop(columns="key")
)

# Compute and store `composition`.
composition = template.merge(
    counts,
    on=[sample_col, condition_col, celltype_col],
    how="left"
)

# Compute and store `composition['n_cells']`.
composition["n_cells"] = composition["n_cells"].fillna(0)

# Calculate `composition['percent']` from the existing values.
composition["percent"] = (
    composition["n_cells"] / composition["total_cells"] * 100
)


In [ ]:
composition


In [ ]:
# Reset the DataFrame index after reshaping or filtering.
composition_wide = composition.pivot(
    index=sample_col,
    columns=celltype_col,
    values="percent"
).reset_index()


In [ ]:
composition_wide


## Selected cell types composition plots

Plot quantitative comparisons across the experimental groups.


In [ ]:
# -----------------------------
# Helper functions
# -----------------------------

# Define `p_to_star()` for reuse in the analysis below.
def p_to_star(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"


# Define `calculate_pairwise_tests()` for reuse in the analysis below.
def calculate_pairwise_tests(
    data,
    celltypes,
    comparisons=[("WT", "R636Q"), ("R636Q", "BE"), ("WT", "BE")],
    test="ttest"
):
    """
    test can be:
    - 'ttest'
    - 'mannwhitney'
    """

    results = []

    for ct in celltypes:
        sub = data[data[celltype_col] == ct]

        for g1, g2 in comparisons:
            x = sub.loc[sub[condition_col] == g1, "percent"].dropna()
            y = sub.loc[sub[condition_col] == g2, "percent"].dropna()

            if len(x) < 2 or len(y) < 2:
                p = np.nan
            else:
                if test == "ttest":
                    _, p = ttest_ind(x, y, equal_var=False)
                elif test == "mannwhitney":
                    _, p = mannwhitneyu(x, y, alternative="two-sided")
                else:
                    raise ValueError("test must be 'ttest' or 'mannwhitney'")

            results.append({
                celltype_col: ct,
                "comparison": f"{g1} vs {g2}",
                "group1": g1,
                "group2": g2,
                "pvalue": p
            })

    results = pd.DataFrame(results)

    # FDR correction across all tests in this plot
    valid = results["pvalue"].notna()

    if valid.sum() > 0:
        results.loc[valid, "padj"] = multipletests(
            results.loc[valid, "pvalue"],
            method="fdr_bh"
        )[1]
    else:
        results["padj"] = np.nan

    results["stars"] = results["padj"].apply(
        lambda p: p_to_star(p) if pd.notna(p) else ""
    )

    return results


# Define `plot_composition()` for reuse in the analysis below.
def plot_composition(
    data,
    celltypes,
    title="Cell-state composition",
    ylimit=None,
    add_stats=True,
    test="ttest",
    comparisons=[("WT", "R636Q"), ("R636Q", "BE")],
    figsize=(8, 5),
    size_scale=1.0
):

    plot_data = data[data[celltype_col].isin(celltypes)].copy()

    fig, ax = plt.subplots(figsize=figsize)

    x_positions = np.arange(len(celltypes))
    width = 0.22

    condition_offsets = {
        condition_order[0]: -width,
        condition_order[1]: 0,
        condition_order[2]: width
    }

    # Plot individual points and mean ± SEM
    for condition in condition_order:
        cond_data = plot_data[plot_data[condition_col] == condition]

        for i, ct in enumerate(celltypes):
            vals = cond_data.loc[cond_data[celltype_col] == ct, "percent"].values
            x = x_positions[i] + condition_offsets[condition]

            # jitter points
            jitter = np.random.normal(0, 0.025, size=len(vals))

            ax.scatter(
                np.repeat(x, len(vals)) + jitter,
                vals,
                color=palette[condition],
                s=30*size_scale,
                alpha=0.9,
                label=condition if i == 0 else None
            )

            if len(vals) > 0:
                mean = np.mean(vals)
                sem = np.std(vals, ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0

                ax.errorbar(
                    x,
                    mean,
                    yerr=sem,
                    color=palette[condition],
                    fmt="_",
                    markersize=18,
                    capsize=4*size_scale,
                    capthick=2*size_scale,
                    elinewidth=2*size_scale
                )

    # Significance annotations
    if add_stats:
        stats = calculate_pairwise_tests(
            plot_data,
            celltypes=celltypes,
            comparisons=comparisons,
            test=test
        )

        y_max = plot_data["percent"].max()
        if ylimit is not None:
            y_max = ylimit[1]

        y_step = y_max * 0.07

        for i, ct in enumerate(celltypes):
            ct_stats = stats[stats[celltype_col] == ct].copy()
            ct_values = plot_data.loc[plot_data[celltype_col] == ct, "percent"]

            base_y = ct_values.max() + y_step if len(ct_values) > 0 else y_step

            for j, row in ct_stats.iterrows():
                if row["stars"] == "ns" or row["stars"] == "":
                    continue

                g1 = row["group1"]
                g2 = row["group2"]

                x1 = x_positions[i] + condition_offsets[g1]
                x2 = x_positions[i] + condition_offsets[g2]
                y = base_y + j % len(comparisons) * y_step

                ax.plot(
                    [x1, x1, x2, x2],
                    [y, y + y_step * 0.2, y + y_step * 0.2, y],
                    color="black",
                    linewidth=1 *size_scale
                )

                ax.text(
                    (x1 + x2) / 2,
                    y + y_step * 0.25,
                    row["stars"],
                    ha="center",
                    va="bottom",
                    fontsize=10 *size_scale
                )

    ax.set_title(title, fontsize=18)
    ax.set_ylabel("Cells per section (%)", fontsize=14)

    ax.set_xticks(x_positions)
    ax.set_xticklabels(celltypes, rotation=40, ha="right", fontsize=12)

    ax.legend(frameon=False, fontsize=12)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if ylimit is not None:
        ax.set_ylim(ylimit)
    ax.grid(False)

    plt.tight_layout()

    return fig, ax


## Statistical comparisons

Perform the statistical tests associated with the plotted group comparisons.


In [ ]:
# Define the values used for `main_celltypes`.
main_celltypes = [
    "Basal VCMs",
    "IFN-associated VCMs",
    "Stressed VCMs",
    "FBs"
]

# Compute and store `(fig, ax)`.
fig, ax = plot_composition(
    data=composition,
    celltypes=main_celltypes,
    title="Cell-state composition",
    ylimit=(-1, 70),
    add_stats=True,
    test="ttest",   # or "mannwhitney"
    comparisons=[("WT", "R636Q"), ("R636Q", "BE"), ("WT", "BE")],
    figsize=(6, 7),
    size_scale=0.65
)

# Save the completed figure to disk.
fig.savefig(
    out_dir/"cell_state_composition.pdf",
    transparent=True,
    bbox_inches="tight"
)

# Display the completed figure.
plt.show()


In [ ]:
# Define the values used for `main_celltypes`.
main_celltypes = [
    "T cells",
    "Myeloid"
]

# Compute and store `(fig, ax)`.
fig, ax = plot_composition(
    data=composition,
    celltypes=main_celltypes,
    title="Cell-state composition",
    ylimit=(-1, 10),
    add_stats=True,
    test="ttest",   # or "mannwhitney"
    comparisons=[("WT", "R636Q"), ("R636Q", "BE"), ("WT", "BE")],
    figsize=(4, 7),
    size_scale=0.65
)
# Save the completed figure to disk.
fig.savefig(
    out_dir/"cell_state_composition_Tcells_myeloid.pdf",
    transparent=True,
    bbox_inches="tight"
)

# Display the completed figure.
plt.show()


## Generate spatial plots with selected cell types highlights

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `selected_cts`.
selected_cts = ["IFN-associated VCMs", "Stressed VCMs", "Basal VCMs", "Remodelled VCMs"]
# Set `highlight_ct` for the following analysis.
highlight_ct = "IFN-associated VCMs"

# subset only BE cells
adata_be = adata[adata.obs["type"] == "BE"].copy()

# sample names
be_samples = list(adata_be.obs["name"].unique())
# Compute and store `be_samples`.
be_samples = sorted(be_samples)   # optional

# get original color of highlighted cell type
cats = list(adata.obs["celltype3"].cat.categories)
# Compute and store `cols`.
cols = list(adata.uns["celltype3_colors"])
# Compute and store `color_map`.
color_map = dict(zip(cats, cols))
# Select the required subset and store it as `highlight_color`.
highlight_color = color_map[highlight_ct]

# figure
fig, axes = plt.subplots(1, len(be_samples), figsize=(5 * len(be_samples), 5))

# Apply the following step only when this condition is met.
if len(be_samples) == 1:
    axes = [axes]

# Repeat the following operation for each item in the selected collection.
for ax, sample in zip(axes, be_samples):
    sub = adata_be[adata_be.obs["name"] == sample].copy()

    # set all non-selected cell types to NA -> plotted as na_color
    celltypes = sub.obs["celltype3"].astype(str)
    sub.obs.loc[~celltypes.isin(selected_cts), "celltype3"] = np.nan

    # plot grey cells first, selected cells second, highlighted cells last
    draw_rank = np.select(
        [
            sub.obs["celltype3"].isna(),
            sub.obs["celltype3"].astype(str).eq(highlight_ct)
        ],
        [0, 2],
        default=1
    )

    sub = sub[np.argsort(draw_rank)].copy()

    # base plot into chosen subplot axis
    sc.pl.spatial(
        sub,
        color="celltype3",
        na_color="lightgrey",
        na_in_legend=False,
        spot_size=25,
        show=False,
        ax=ax
    )

    # overlay highlighted cells again to make them more visible
    mask_highlight = sub.obs["celltype3"].astype(str).eq(highlight_ct).values
    coords = sub.obsm["spatial"][mask_highlight]

    ax.scatter(
        coords[:, 0],
        coords[:, 1],
        s=3,   # increase if you want them thicker
        c=highlight_color,
        edgecolors="none"
    )

    ax.set_title(sample)

    # remove frame / borders
    ax.set_frame_on(False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    # remove ticks and axis labels
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    # add scalebar
    scalebar_um = 1000

    # if coordinates are in microns:
    scalebar_length = scalebar_um

    # if coordinates are in pixels instead, use this instead:
    # px_per_um = 1 / 0.2125
    # scalebar_length = scalebar_um * px_per_um

    fontprops = FontProperties(size=10)

    scalebar = AnchoredSizeBar(
        ax.transData,
        scalebar_length,
        f"{scalebar_um} µm",
        loc="lower right",
        pad=0.3,
        color="black",
        frameon=False,
        size_vertical=scalebar_length * 0.01,
        fontproperties=fontprops
    )

    ax.add_artist(scalebar)

# Adjust figure spacing to avoid overlapping elements.
plt.tight_layout()

# Save the completed figure to disk.
plt.savefig(
    out_dir / "BE_all_samples_VCMs_highlighted_panel.pdf",
    bbox_inches="tight",
    dpi=600
)

# Display the completed figure.
plt.show()


In [ ]:
# Load the processed AnnData object.
adata = sc.read_h5ad("../../combined_Xenium_largecells_noWT3_afterclustering_withleiden_annotated_ordered_withTcells_ReannotatedIfgga4_newcolours.h5ad")


In [ ]:
# Make an independent copy of the selected data.
BE_rep2 = adata[adata.obs['name'] == 'BE_rep2'].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `target_cts`.
target_cts = [
    "IFN-associated VCMs",
    "T cells",
    "Myeloid"
]

# Compute and store `plot_order`.
plot_order = np.argsort(
    BE_rep2.obs["celltype3"].isin(target_cts).values
)

# Make an independent copy of the selected data.
BE_rep2_plot = BE_rep2[plot_order].copy()

# ------------------------------------------------------------
# Fixed plotting dimensions
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (5500, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Create figure explicitly
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Spatial plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    BE_rep2_plot,
    color="celltype3",
    na_color="lightgrey",
    spot_size=25,
    ax=ax,
    show=False
)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Fixed scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=12)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.4,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=10,
    fontproperties=fontprops
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Clean axes
# ------------------------------------------------------------

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Keep margins consistent between samples
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "BE_rep2_targets_on_top_IfggaTMy_scalebar.pdf",
    dpi=600
)

# Display the completed figure.
plt.show()


In [ ]:
# Make an independent copy of the selected data.
BE_rep4 = adata[adata.obs['name'] == 'BE_rep4'].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `target_cts`.
target_cts = [
    "IFN-associated VCMs",
    "T cells",
    "Myeloid"
]

# Compute and store `plot_order`.
plot_order = np.argsort(
    BE_rep4.obs["celltype3"].isin(target_cts).values
)

# Make an independent copy of the selected data.
BE_rep4_plot = BE_rep4[plot_order].copy()

# ------------------------------------------------------------
# Fixed plotting dimensions
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (5500, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Create figure explicitly
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Spatial plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    BE_rep4_plot,
    color="celltype3",
    na_color="lightgrey",
    spot_size=25,
    ax=ax,
    show=False
)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Fixed scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=12)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.4,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=10,
    fontproperties=fontprops
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Clean axes
# ------------------------------------------------------------

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Keep margins consistent between samples
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "BE_rep4_targets_on_top_IfggaTMy_scalebar.pdf",
    dpi=600
)

# Display the completed figure.
plt.show()


In [ ]:
# Make an independent copy of the selected data.
BE_rep3 = adata[adata.obs['name'] == 'BE_rep3'].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `target_cts`.
target_cts = [
    "IFN-associated VCMs",
    "T cells",
    "Myeloid"
]

# Compute and store `plot_order`.
plot_order = np.argsort(
    BE_rep3.obs["celltype3"].isin(target_cts).values
)

# Make an independent copy of the selected data.
BE_rep3_plot = BE_rep3[plot_order].copy()

# ------------------------------------------------------------
# Fixed plotting dimensions
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (5500, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Create figure explicitly
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Spatial plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    BE_rep3_plot,
    color="celltype3",
    na_color="lightgrey",
    spot_size=25,
    ax=ax,
    show=False
)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Fixed scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=12)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.4,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=10,
    fontproperties=fontprops
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Clean axes
# ------------------------------------------------------------

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Keep margins consistent between samples
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "BE_rep3_targets_on_top_IfggaTMy_scalebar.pdf",
    dpi=600
)

# Display the completed figure.
plt.show()


In [ ]:
# Make an independent copy of the selected data.
BE_rep1 = adata[adata.obs['name'] == 'BE_rep1'].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `target_cts`.
target_cts = [
    "IFN-associated VCMs",
    "T cells",
    "Myeloid"
]

# Compute and store `plot_order`.
plot_order = np.argsort(
    BE_rep1.obs["celltype3"].isin(target_cts).values
)

# Make an independent copy of the selected data.
BE_rep1_plot = BE_rep1[plot_order].copy()

# ------------------------------------------------------------
# Fixed plotting dimensions
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (5500, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Create figure explicitly
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Spatial plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    BE_rep1_plot,
    color="celltype3",
    na_color="lightgrey",
    spot_size=25,
    ax=ax,
    show=False
)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Fixed scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=12)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.4,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=10,
    fontproperties=fontprops
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Clean axes
# ------------------------------------------------------------

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Keep margins consistent between samples
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "BE_rep1_targets_on_top_IfggaTMy_scalebar.pdf",
    dpi=600
)

# Display the completed figure.
plt.show()


In [ ]:
# Make an independent copy of the selected data.
WT_rep5 = adata[adata.obs['name'] == 'WT_rep5'].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `target_cts`.
target_cts = [
    "IFN-associated VCMs",
    "T cells",
    "Myeloid"
]

# Compute and store `plot_order`.
plot_order = np.argsort(
    WT_rep5.obs["celltype3"].isin(target_cts).values
)

# Make an independent copy of the selected data.
WT_rep5_plot = WT_rep5[plot_order].copy()

# ------------------------------------------------------------
# Fixed plotting dimensions
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (6000, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Create figure explicitly
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Spatial plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    WT_rep5_plot,
    color="celltype3",
    na_color="lightgrey",
    spot_size=25,
    ax=ax,
    show=False
)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Fixed scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=12)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.4,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=10,
    fontproperties=fontprops
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Clean axes
# ------------------------------------------------------------

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Keep margins consistent between samples
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "WT_rep5_targets_on_top_IfggaTMy_scalebar.pdf",
    dpi=600
)

# Display the completed figure.
plt.show()


In [ ]:
# Make an independent copy of the selected data.
WT_rep1 = adata[adata.obs['name'] == 'WT_rep1'].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `target_cts`.
target_cts = [
    "IFN-associated VCMs",
    "T cells",
    "Myeloid"
]

# Compute and store `plot_order`.
plot_order = np.argsort(
    WT_rep1.obs["celltype3"].isin(target_cts).values
)

# Make an independent copy of the selected data.
WT_rep1_plot = WT_rep1[plot_order].copy()

# ------------------------------------------------------------
# Fixed plotting dimensions
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (6000, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Create figure explicitly
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Spatial plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    WT_rep1_plot,
    color="celltype3",
    na_color="lightgrey",
    spot_size=25,
    ax=ax,
    show=False
)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Fixed scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=12)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.4,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=10,
    fontproperties=fontprops
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Clean axes
# ------------------------------------------------------------

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Keep margins consistent between samples
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "WT_rep1_targets_on_top_IfggaTMy_scalebar.pdf",
    dpi=600
)

# Display the completed figure.
plt.show()


In [ ]:
# Make an independent copy of the selected data.
WT_rep4 = adata[adata.obs['name'] == 'WT_rep4'].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `target_cts`.
target_cts = [
    "IFN-associated VCMs",
    "T cells",
    "Myeloid"
]

# Compute and store `plot_order`.
plot_order = np.argsort(
    WT_rep4.obs["celltype3"].isin(target_cts).values
)

# Make an independent copy of the selected data.
WT_rep4_plot = WT_rep4[plot_order].copy()

# ------------------------------------------------------------
# Fixed plotting dimensions
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (6000, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Create figure explicitly
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Spatial plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    WT_rep4_plot,
    color="celltype3",
    na_color="lightgrey",
    spot_size=25,
    ax=ax,
    show=False
)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Fixed scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=12)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.4,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=10,
    fontproperties=fontprops
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Clean axes
# ------------------------------------------------------------

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Keep margins consistent between samples
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "WT_rep4_targets_on_top_IfggaTMy_scalebar.pdf",
    dpi=600
)

# Display the completed figure.
plt.show()


In [ ]:
# Make an independent copy of the selected data.
WT_rep2 = adata[adata.obs['name'] == 'WT_rep2'].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `target_cts`.
target_cts = [
    "IFN-associated VCMs",
    "T cells",
    "Myeloid"
]

# Compute and store `plot_order`.
plot_order = np.argsort(
    WT_rep2.obs["celltype3"].isin(target_cts).values
)

# Make an independent copy of the selected data.
WT_rep2_plot = WT_rep2[plot_order].copy()

# ------------------------------------------------------------
# Fixed plotting dimensions
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (6000, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Create figure explicitly
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Spatial plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    WT_rep2_plot,
    color="celltype3",
    na_color="lightgrey",
    spot_size=25,
    ax=ax,
    show=False
)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Fixed scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=12)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.4,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=10,
    fontproperties=fontprops
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Clean axes
# ------------------------------------------------------------

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Keep margins consistent between samples
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "WT_rep2_targets_on_top_IfggaTMy_scalebar.pdf",
    dpi=600
)

# Display the completed figure.
plt.show()


In [ ]:
# Make an independent copy of the selected data.
PBS_rep2 = adata[adata.obs['name'] == 'PBS_rep2'].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `target_cts`.
target_cts = [
    "IFN-associated VCMs",
    "T cells",
    "Myeloid"
]

# Compute and store `plot_order`.
plot_order = np.argsort(
    PBS_rep2.obs["celltype3"].isin(target_cts).values
)

# Make an independent copy of the selected data.
PBS_rep2_plot = PBS_rep2[plot_order].copy()

# ------------------------------------------------------------
# Fixed plotting dimensions
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (6000, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Create figure explicitly
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Spatial plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    PBS_rep2_plot,
    color="celltype3",
    na_color="lightgrey",
    spot_size=25,
    ax=ax,
    show=False
)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Fixed scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=12)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.4,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=10,
    fontproperties=fontprops
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Clean axes
# ------------------------------------------------------------

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Keep margins consistent between samples
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "PBS_rep2_targets_on_top_IfggaTMy_scalebar.pdf",
    dpi=600
)

# Display the completed figure.
plt.show()


In [ ]:
# Make an independent copy of the selected data.
PBS_rep3 = adata[adata.obs['name'] == 'PBS_rep3'].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `target_cts`.
target_cts = [
    "IFN-associated VCMs",
    "T cells",
    "Myeloid"
]

# Compute and store `plot_order`.
plot_order = np.argsort(
    PBS_rep3.obs["celltype3"].isin(target_cts).values
)

# Make an independent copy of the selected data.
PBS_rep3_plot = PBS_rep3[plot_order].copy()

# ------------------------------------------------------------
# Fixed plotting dimensions
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (6000, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Create figure explicitly
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Spatial plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    PBS_rep3_plot,
    color="celltype3",
    na_color="lightgrey",
    spot_size=25,
    ax=ax,
    show=False
)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Fixed scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=12)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.4,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=10,
    fontproperties=fontprops
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Clean axes
# ------------------------------------------------------------

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Keep margins consistent between samples
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "PBS_rep3_targets_on_top_IfggaTMy_scalebar.pdf",
    dpi=600
)

# Display the completed figure.
plt.show()


In [ ]:
# Make an independent copy of the selected data.
PBS_rep1 = adata[adata.obs['name'] == 'PBS_rep1'].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `target_cts`.
target_cts = [
    "IFN-associated VCMs",
    "T cells",
    "Myeloid"
]

# Compute and store `plot_order`.
plot_order = np.argsort(
    PBS_rep1.obs["celltype3"].isin(target_cts).values
)

# Make an independent copy of the selected data.
PBS_rep1_plot = PBS_rep1[plot_order].copy()

# ------------------------------------------------------------
# Fixed plotting dimensions
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (6000, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Create figure explicitly
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Spatial plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    PBS_rep1_plot,
    color="celltype3",
    na_color="lightgrey",
    spot_size=25,
    ax=ax,
    show=False
)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Fixed scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=12)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.4,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=10,
    fontproperties=fontprops
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Clean axes
# ------------------------------------------------------------

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Keep margins consistent between samples
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "PBS_rep1_targets_on_top_IfggaTMy_scalebar.pdf",
    dpi=600
)

# Display the completed figure.
plt.show()


In [ ]:
# Make an independent copy of the selected data.
PBS_rep4 = adata[adata.obs['name'] == 'PBS_rep4'].copy()


In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

# Define the values used for `target_cts`.
target_cts = [
    "IFN-associated VCMs",
    "T cells",
    "Myeloid"
]

# Compute and store `plot_order`.
plot_order = np.argsort(
    PBS_rep4.obs["celltype3"].isin(target_cts).values
)

# Make an independent copy of the selected data.
PBS_rep4_plot = PBS_rep4[plot_order].copy()

# ------------------------------------------------------------
# Fixed plotting dimensions
# Use exactly the same values for every sample
# ------------------------------------------------------------

# Define the values used for `common_xlim`.
common_xlim = (-500, 5500)
# Define the values used for `common_ylim`.
common_ylim = (6000, -500)

# Define the values used for `figure_size`.
figure_size = (8, 8)

# Create figure explicitly
fig, ax = plt.subplots(figsize=figure_size)

# ------------------------------------------------------------
# Spatial plot
# ------------------------------------------------------------

# Plot the selected feature in spatial coordinates.
sc.pl.spatial(
    PBS_rep4_plot,
    color="celltype3",
    na_color="lightgrey",
    spot_size=25,
    ax=ax,
    show=False
)

# ------------------------------------------------------------
# Fixed spatial scaling
# ------------------------------------------------------------

# Run `ax.set_xlim` for this analysis step.
ax.set_xlim(common_xlim)
# Run `ax.set_ylim` for this analysis step.
ax.set_ylim(common_ylim)
# Run `ax.set_aspect` for this analysis step.
ax.set_aspect("equal", adjustable="box")

# ------------------------------------------------------------
# Fixed scale bar
# ------------------------------------------------------------

# Set `scalebar_um` for the following analysis.
scalebar_um = 1000

# Compute and store `fontprops`.
fontprops = FontProperties(size=12)

# Compute and store `scalebar`.
scalebar = AnchoredSizeBar(
    ax.transData,
    scalebar_um,
    f"{scalebar_um} µm",
    loc="lower right",
    pad=0.4,
    borderpad=0.5,
    sep=4,
    color="black",
    frameon=False,
    size_vertical=10,
    fontproperties=fontprops
)

# Run `ax.add_artist` for this analysis step.
ax.add_artist(scalebar)

# ------------------------------------------------------------
# Clean axes
# ------------------------------------------------------------

# Run `ax.set_xticks` for this analysis step.
ax.set_xticks([])
# Run `ax.set_yticks` for this analysis step.
ax.set_yticks([])

# Keep margins consistent between samples
fig.subplots_adjust(
    left=0.05,
    right=0.82,
    bottom=0.05,
    top=0.92,
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

# Save the completed figure to disk.
fig.savefig(
    out_dir / "PBS_rep4_targets_on_top_IfggaTMy_scalebar.pdf",
    dpi=600
)

# Display the completed figure.
plt.show()
